In [2]:
%run Packages.ipynb
%run Graph_Generators.ipynb
%run Cycle_Mtx_functions.ipynb
# from random import sample


In [22]:
def verifier(G,S):
    """Verify if S is a light cover of G. If so, return a valid solution."""
    H = G.copy()
    M = max(G.edge_labels())
    for e in S:
        H.set_edge_label(e[0],e[1],M)
    D = H.distance_all_pairs(by_weight=True)
    for u,v,w in H.edges(sort=True):
        if D[u][v] != w:
            H.set_edge_label(u,v,D[u][v])
            if (u,v) not in S and (v,u) not in S:
                return 0
    return 1

### Heuristics ###

def Gilbert_Jain_IOMR(Kn):
    """The Gilbert & Lalit heuristic - just arbitrarily pick and edge to fix in a broken triangle.
    Assumes input is Complete."""
    H = Kn.copy()
    M = H.weighted_adjacency_matrix()
    n = M.nrows()
    S = set()
    for k in range(n):
        for i in range(n):
            for j in range(i):
                if M[i,j] > M[i,k] + M[k,j]:
                    M[i,k] = M[i,j] - M[k,j]
                    S.add(tuple(sorted((i,k))))

    return S


def MVD_Pivot_Rec(ind,x,S,Kn):
    """I will think of x as the symmetric adjacency matrix and M as the fixed copy of the 
     adjacency matrix which we augment along the way"""
    if len(ind) <=2:
        return
    i = np.random.choice(ind)
    
    ind_i = ind.copy() #make sure we don't change the input
    ind_i.remove(i) # Remove the current pivot
    
    for j,k in list(combinations(ind_i,2)): #iterate over all triangles i,j,k
        ADD = False #flag - do we change the edge?
        if x[j,k] > x[i,j] + x[i,k]:
            x[j,k] = x[i,j] + x[i,k]
            ADD=True
        if x[j,k] < np.abs(x[i,j] - x[i,k]):
            x[j,k] = np.abs(x[i,j] - x[i,k])
            ADD=True
        if ADD:
            S.add(tuple(sorted((j,k))))
            
    MVD_Pivot_Rec(ind_i,x,S,Kn)
    return
        
def MVD_Pivot(Kn):
    """The algorithm from Fitting Metrics with Minimum Disagreement. This one solves a general MR, not IOMR"""
    A = Kn.weighted_adjacency_matrix()
    S = set()
    MVD_Pivot_Rec(list(range(Kn.num_verts())),A,S,Kn)
    T = np.triu(A)
    return S


def reduce_solution(S,G,Diff = []):
    """Takes a graph G and an extended HS S and reduces it to a HS for G (discards edges that are not in G).
    Diff is a matrix that allows to check if an edge was actually changed if needed"""
#     print(len(S))
    reduced_S = set()
    for e in G.edges(labels=False, sort=True): #check all edges
        if e in S:
            reduced_S.add(e) #if it hits a broken cycle, add it
    return reduced_S
    


#### This is basically done now, just need to run some tests and then actually run it.
def l1_min_heuristic(G):
    """Solves metric repair by creating an l1 minimization problem. That is:
    1. Complete the graph G, obtain Gc
    2. Create the metric constraint matrix Phi, which checks all triangle inequalities in Gc
    3. Solve L1 minimization, guaranteed to satisfy all triangle inequalities
    4. Take the intersection of E(G) with the support of the solution. Since no broken triangles remain, that means
    no broken cycles in general remain, so the intersection must be a light cover of G"""
    
    ## Preprocess - comlplete the graph and get index dictionaries
    Gc = complete(G)
    D = make_index_encoding(Gc)
    w = get_weights_vector(Gc,D)
    
    ## Set up the minimization problem
    phi = induc(Gc)
    Phi = np.transpose(phi)
    m,n = Phi.shape
    b = np.transpose(Phi) @ w
    c = np.ones(m)
    con = -np.transpose(Phi)
    
    ## Solve minimization problem
    soln = linprog(c,A_ub = con, b_ub = b, method = 'simplex') 
    
    ## Retrieve the support of solution
    S = set()
    for e in G.edges(sort=True):
        if soln.x[D[(e[0],e[1])]] > 0:
            S.add((e[0],e[1]))
    return S

def l1_minimization(Gc):
    """ The same as l1 min but without completion """
    phi = induced_cycle_matrix(Gc)
    if phi[1] == 0:
        return set()
    else:
        phi = phi[0]
    Phi = np.transpose(phi)
    D = make_index_encoding(Gc)
    w = get_weights_vector(Gc,D)
    # print(Gc.edges(sort=True))
    m,n = Phi.shape
    b = np.transpose(Phi) @ w
    c = np.ones(m)
    con = -np.transpose(Phi)
    soln = linprog(c,A_ub = con, b_ub = b, method = 'highs') 
    x = soln.x
    S = set()
    for e in Gc.edges(sort=True):
        if x[D[(e[0],e[1])]] > 0:
            S.add((e[0],e[1]))
    return S

def domr_alg(G):
    S = set()
    apsp = G.distance_all_pairs(by_weight=True)
    for e in G.edges(sort=True):
        if e[2] != apsp[e[0]][e[1]]:
            S.add((e[0],e[1]))
    return(S)


def left_edge_heuristic(G):
    Kn = complete(G)
    S = Gilbert_Jain_IOMR(Kn)
    return reduce_solution(S,G)
    

def pivot_heuristic(G):
    Kn = complete(G)
    S = MVD_Pivot(Kn)
    return reduce_solution(S,G)

def find_shortest_path(u,v,Dict):
    """
    Given a dictionary of predecessors, find a shortest path between u,v
    """
    start = u
    end = v
    P = list()
    while end != u:
        w = Dict[u][end] ## find a predecessor of end
        P.append(tuple(sorted((end,w)))) #add that edge to the list
        end = w #w is new endpoint
    return P


### Gives an L+1 (or L) approximation for Graph Metric Repair
def shortest_path_cover(G,general = True):
    H = G.copy()
    D = get_weights(G)
#     A = H.weighted_adjacency_matrix()
    S = set()
    while True:
        dist, Dict = floyd_warshall_shortest_paths(H,predecessors=1,distances=1)
        found_broken = 0
        for u,v,w in H.edges(sort=True):
            if D[(u,v)] > dist[u][v]:
                found_broken = 1
                P = find_shortest_path(u,v,Dict)
                if general:
                    S.add((tuple(sorted((u,v)))))
                    H.delete_edge((u,v))
                S.update(P)
                H.delete_edges(P)
        if not found_broken:
            return S